# Job Market & Skills Analysis

## Research Question

**What do analyst, product, and business-intelligence job postings actually ask for, and how do role family, experience level, salary, and tool requirements differ across them?**

This notebook follows the same workflow used in my coursework: question → data overview → cleaning → exploratory analysis → interpretation → limitations.


### Importing


In [ ]:
import re
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams['figure.figsize'] = (10, 6)


### Data overview

The repository contains job postings plus related salary, skill, company, and industry tables. The main analysis starts from `postings.csv` and uses only the columns needed for the research question.


In [ ]:
POSTING_COLUMNS = ['job_id','title','description','location','formatted_experience_level','normalized_salary','remote_allowed','formatted_work_type']
postings = pd.read_csv('postings.csv', usecols=POSTING_COLUMNS)
print('Rows:', len(postings))
postings.head()


# Data Cleaning Appendix

The title classifier is intentionally conservative: a posting is assigned to a role family only when the title contains a clear matching phrase.


In [ ]:
def classify_role(title):
    title = str(title).lower()
    if re.search(r'\\b(product manager|associate product manager|product management)\\b', title): return 'Product management'
    if re.search(r'\\bproduct analyst\\b', title): return 'Product analyst'
    if re.search(r'\\b(business intelligence|bi analyst|business intelligence analyst)\\b', title): return 'Business intelligence'
    if re.search(r'\\b(business systems analyst|business system analyst|business analyst)\\b', title): return 'Business analyst'
    if re.search(r'\\b(data analyst|analytics analyst|data analytics analyst)\\b', title): return 'Data analyst'
    return pd.NA

postings['role_family'] = postings['title'].apply(classify_role)
target = postings.dropna(subset=['role_family']).copy()
target['formatted_experience_level'] = target['formatted_experience_level'].fillna('Missing').replace('', 'Missing')
target['normalized_salary'] = pd.to_numeric(target['normalized_salary'], errors='coerce')
target.loc[target['normalized_salary'] <= 0, 'normalized_salary'] = pd.NA
print('All postings:', len(postings))
print('Target-role postings:', len(target))


In [ ]:
TOOL_PATTERNS = {'SQL':r'\\bsql\\b','Excel':r'\\bexcel\\b','Python':r'\\bpython\\b','Tableau':r'\\btableau\\b','Power BI':r'\\bpower\\s*bi\\b','R':r'\\bR\\b','A/B testing':r'\\ba/b test(?:ing)?\\b','Jira':r'\\bjira\\b','Figma':r'\\bfigma\\b'}
for tool, pattern in TOOL_PATTERNS.items():
    flags = 0 if tool == 'R' else re.IGNORECASE
    target[tool] = target['description'].fillna('').str.contains(pattern, regex=True, flags=flags)


# Exploratory Analysis

## 1. Role-family distribution

![Role volume](results/charts/role_volume.svg)

Business Analyst (67) and Product Management (59) are the largest groups, followed by Data Analyst (45), Business Intelligence (15), and Product Analyst (2).


In [ ]:
role_counts = target['role_family'].value_counts().sort_values()
role_counts.plot(kind='barh', title='Target-role postings in the dataset')
plt.xlabel('Postings'); plt.ylabel(''); plt.tight_layout(); plt.show()
role_counts


## 2. Experience-level distribution

![Experience distribution](results/charts/experience_distribution.svg)

Mid-Senior level is the most common explicit label. Data Analyst is the most early-career-heavy category in this sample, with 19 of 45 postings labeled Entry level.


In [ ]:
experience_counts = target['formatted_experience_level'].value_counts().sort_values()
experience_counts.plot(kind='barh', title='Experience labels across target-role postings')
plt.xlabel('Postings'); plt.ylabel(''); plt.tight_layout(); plt.show()
pd.crosstab(target['role_family'], target['formatted_experience_level'])


## 3. Salary by role family

![Salary by role](results/charts/salary_by_role.svg)

Salary coverage is incomplete. Product Management has the highest observed median among postings with usable salary data, but it also contains a more senior experience mix.


In [ ]:
salary_summary = target.dropna(subset=['normalized_salary']).groupby('role_family')['normalized_salary'].agg(['count','median']).sort_values('median')
salary_summary['median'].plot(kind='barh', title='Median normalized salary where salary data is available')
plt.xlabel('Normalized annual salary (USD)'); plt.ylabel(''); plt.tight_layout(); plt.show()
salary_summary


## 4. Tool and skill mentions

![Skill keywords](results/charts/skill_keywords.svg)

SQL is the most common keyword in the target sample (55 postings, 29.3%), followed by Excel (43, 22.9%), Python (23, 12.2%), Tableau (22, 11.7%), and Power BI (20, 10.6%).


In [ ]:
tool_summary = pd.DataFrame({'count': target[list(TOOL_PATTERNS)].sum()})
tool_summary['share_pct'] = tool_summary['count'] / len(target) * 100
tool_summary.sort_values('share_pct')['share_pct'].plot(kind='barh', title='Tool and skill keyword mentions')
plt.xlabel('Share of target postings (%)'); plt.ylabel(''); plt.tight_layout(); plt.show()
target.groupby('role_family')[list(TOOL_PATTERNS)].mean().mul(100).round(1)


# Findings

1. **Seniority mix differs substantially by role family.** Data Analyst has a much larger Entry-level share than Product Management or Business Analyst in this sample.
2. **SQL is the strongest recurring technical signal across adjacent roles.**
3. **Salary comparisons require seniority context.** Product Management has the highest observed median salary, but also a more senior posting mix.
4. **Adjacent roles overlap without being identical.** BI descriptions are especially dense in SQL, Excel, Power BI, and Tableau mentions.


# Data Description

### Motivation

The analysis began with a practical observation: adjacent role titles can hide very different expectations.

### Composition

The repository contains job-level postings plus related tables for companies, industries, skills, and salaries.

### Collection process

The notebook analyzes the CSV files contained in this repository. It does not attempt to model LinkedIn's ranking or recommendation systems.


# Data Limitations

- This is a snapshot of postings, not a complete census of the current labor market.
- Title classification is rule-based and intentionally conservative.
- 48 of the 188 target postings have no experience label.
- Salary coverage is incomplete and mixes seniority levels.
- Keyword counts detect literal mentions, not proficiency or importance.
- Product Analyst has only two captured postings.
- The analysis is descriptive and does not establish causation.
